In [66]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

# 数据接口：只读写与计算，不输出任何图表

def load_data(path='customer_clusters_simple.csv'):
    """从 CSV 加载用户特征数据并返回 DataFrame。抛出异常由调用方处理。"""
    df = pd.read_csv(path)
    return df


def create_individual_profiles(df: pd.DataFrame) -> pd.DataFrame:
    """将原始特征表标准化为统一的用户画像表格。返回新的 DataFrame。
    兼容多种列名，尽量使用 .get 以避免 KeyError。
    """
    profiles = []

    for _, row in df.iterrows():
        # 安全地读取数值：只在值为 None 或 NaN 时才使用默认值，不用 "or" 避免把 0 当成缺失
        def _safe_float(v, default=0.0):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                return float(default)
            return float(v)

        def _safe_int(v, default=0):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                return int(default)
            return int(v)

        total_sales = _safe_float(row.get('Total_Sales', 0.0), 0.0)
        total_orders = _safe_int(row.get('Total_Orders', 0), 0)
        avg_order_value = _safe_float(row.get('Avg_Order_Value', 0.0), 0.0)
        lifetime_val = row.get('Customer_Lifetime_Days', row.get('Customer_Lifetime', 0))
        customer_lifetime = _safe_float(lifetime_val, 0.0)
        purchase_freq = _safe_float(row.get('Purchase_Frequency', 0.0), 0.0)
        profit_margin = _safe_float(row.get('Profit_Margin', 0.0), 0.0)

        last_purchase_raw = row.get('Days_Since_Last_Purchase', row.get('Last_Purchase_Days_Ago', np.nan))
        # 对最近一次购买天数：NaN 才当缺失，0 是“刚购买”
        if last_purchase_raw is None or (isinstance(last_purchase_raw, float) and np.isnan(last_purchase_raw)):
            last_purchase_days_ago = np.nan
        else:
            last_purchase_days_ago = float(last_purchase_raw)

        engagement_raw = row.get('Total_Engagement_Score', row.get('Engagement_Score', 0.0))
        engagement_score = _safe_float(engagement_raw, 0.0)

        unique_products_raw = row.get('Unique_Products_Purchased', row.get('Unique_Products_Bought', 0))
        unique_products_bought = _safe_int(unique_products_raw, 0)

        profile = {
            'Customer_ID': row.get('Customer ID', row.get('Customer_ID', None)),
            'Customer_Name': row.get('Customer Name', row.get('Customer_Name', 'Unknown')),
            'Cluster': row.get('Cluster', None),

            'Total_Sales': total_sales,
            'Total_Orders': total_orders,
            'Avg_Order_Value': avg_order_value,
            'Customer_Lifetime': customer_lifetime,
            'Purchase_Frequency': purchase_freq,
            'Profit_Margin': profit_margin,
            'Last_Purchase_Days_Ago': last_purchase_days_ago,
            'Engagement_Score': engagement_score,

            'Segment': row.get('Segment', None),
            'Region': row.get('Region', None),
            'Country': row.get('Country', None),
            'Gender': row.get('Gender', None),
            'Age': row.get('Age', np.nan),
            'Education': row.get('Education', None),
            'Marital_Status': row.get('Marital Status', row.get('Marital_Status', None)),

            'Favorite_Product': row.get('Favorite_Product', 'Unknown'),
            'Unique_Products_Bought': unique_products_bought,
        }
        profiles.append(profile)

    return pd.DataFrame(profiles)


def calculate_percentiles_and_rankings(df: pd.DataFrame, metrics=None) -> pd.DataFrame:
    """为指定指标计算百分位、排名、评级和与均值对比。返回新 DataFrame。"""
    result = df.copy()
    if metrics is None:
        metrics = [
            'Total_Sales', 'Total_Orders', 'Avg_Order_Value',
            'Purchase_Frequency', 'Profit_Margin', 'Engagement_Score', 'Customer_Lifetime'
        ]

    def get_grade(percentile):
        if percentile >= 80:
            return 'A'
        elif percentile >= 60:
            return 'B'
        elif percentile >= 40:
            return 'C'
        elif percentile >= 20:
            return 'D'
        else:
            return 'E'

    for metric in metrics:
        if metric in result.columns:
            result[f'{metric}_Percentile'] = result[metric].rank(pct=True) * 100
            result[f'{metric}_Rank'] = result[metric].rank(ascending=False, method='min')
            result[f'{metric}_Grade'] = result[f'{metric}_Percentile'].apply(get_grade)
            overall_mean = result[metric].mean()
            # 避免除以零
            if overall_mean != 0 and not np.isnan(overall_mean):
                result[f'{metric}_vs_Mean'] = ((result[metric] - overall_mean) / overall_mean * 100).round(1)
            else:
                result[f'{metric}_vs_Mean'] = 0.0

    return result


def calculate_adjusted_score(value, series, is_reverse=False):
    """用于对单个值根据样本序列计算平滑得分（0-100）。"""
    series = np.array(series.dropna()) if hasattr(series, 'dropna') else np.array(series)
    if len(series) == 0:
        return 0.0
    if is_reverse:
        percentile = (series < value).sum() / len(series) * 100
    else:
        percentile = (series <= value).sum() / len(series) * 100
    score = 100 / (1 + np.exp(-0.1 * (percentile - 50)))
    return float(min(100, max(0, score)))


def create_personalized_description(row) -> str:
    """为单个用户生成文本描述（不打印）。"""
    descriptions = []
    try:
        descriptions.append(f"客户 {row.get('Customer_Name', 'Unknown')} ({row.get('Customer_ID', '')}) 属于第 {row.get('Cluster', '')} 类用户群体。")
        ts_pct = row.get('Total_Sales_Percentile', 0)
        if ts_pct >= 80:
            descriptions.append("高价值客户，总消费金额排在前20%。")
        elif ts_pct >= 60:
            descriptions.append("中高价值客户，消费金额较为可观。")
        elif ts_pct >= 40:
            descriptions.append("中等价值客户。")
        else:
            descriptions.append("低价值客户，有较大的提升空间。")

        if row.get('Customer_Lifetime', 0) > 365:
            descriptions.append("客户关系已维持超过1年，属于长期忠实客户。")
        elif row.get('Customer_Lifetime', 0) > 180:
            descriptions.append("客户关系已维持半年以上，忠诚度较高。")

        last_days = row.get('Last_Purchase_Days_Ago', np.inf)
        if last_days <= 30:
            descriptions.append("近期活跃，最近30天内有过购买。")
        elif last_days <= 90:
            descriptions.append("较为活跃，最近90天内有过购买。")
        elif last_days <= 180:
            descriptions.append("活跃度一般，最近半年内有过购买。")
        else:
            descriptions.append("活跃度较低，需要重新激活。")

        if row.get('Avg_Order_Value_Percentile', 0) >= 80:
            descriptions.append("购买力强，平均订单价值较高。")
        if row.get('Purchase_Frequency_Percentile', 0) >= 80:
            descriptions.append("购买频率高，复购行为明显。")
        if row.get('Engagement_Score_Percentile', 0) >= 80:
            descriptions.append("互动积极，经常浏览、点赞和分享产品。")
        if row.get('Favorite_Product') and row.get('Favorite_Product') != 'Unknown':
            descriptions.append(f"最常购买的产品是：{row.get('Favorite_Product')}")
        if row.get('Unique_Products_Bought', 0) >= 5:
            descriptions.append("产品涉猎广泛，购买过多种不同类型的产品。")

        suggestions = []
        if last_days > 90:
            suggestions.append("发送再营销邮件和专属优惠")
        if ts_pct >= 80:
            suggestions.append("提供VIP专属服务和优先配送")
        if row.get('Purchase_Frequency_Percentile', 0) >= 80:
            suggestions.append("推荐忠诚度计划和定期购服务")
        if row.get('Engagement_Score_Percentile', 0) >= 80:
            suggestions.append("邀请参与产品评测和新品试用")
        if suggestions:
            descriptions.append(f"营销建议：{'；'.join(suggestions)}。")
    except Exception:
        return ''
    return ' '.join(descriptions)


def create_user_analysis_data(user_row: pd.Series, all_users_df: pd.DataFrame) -> dict:
    """为单个用户构建可机器读取的分析数据字典（不打印、不绘图）。"""
    data = {}
    data['Customer_ID'] = user_row.get('Customer_ID')
    data['Customer_Name'] = user_row.get('Customer_Name')
    data['Cluster'] = user_row.get('Cluster')

    # 关键指标
    data['Total_Sales'] = user_row.get('Total_Sales', 0.0)
    data['Total_Orders'] = user_row.get('Total_Orders', 0)
    data['Avg_Order_Value'] = user_row.get('Avg_Order_Value', 0.0)
    data['Profit_Margin'] = user_row.get('Profit_Margin', 0.0)

    # 行为
    data['Customer_Lifetime'] = user_row.get('Customer_Lifetime', 0)
    data['Last_Purchase_Days_Ago'] = user_row.get('Last_Purchase_Days_Ago', np.nan)
    data['Purchase_Frequency'] = user_row.get('Purchase_Frequency', 0.0)
    data['Engagement_Score'] = user_row.get('Engagement_Score', 0.0)

    # 排名与百分位（如果已经计算过）
    for col in list(user_row.index):
        if col.endswith('_Percentile') or col.endswith('_Rank') or col.endswith('_Grade') or col.endswith('_vs_Mean'):
            data[col] = user_row.get(col)

    # 文本描述
    data['Personalized_Description'] = create_personalized_description(user_row)

    return data


def save_all_user_profiles(profiles_df: pd.DataFrame, output_file='all_user_profiles_detailed.csv') -> str:
    """保存用户画像到 CSV，返回保存路径。"""
    existing_columns = [col for col in profiles_df.columns]
    profiles_df.to_csv(output_file, index=False, encoding='utf-8')
    return output_file

# 导出符号列表
__all__ = [
    'load_data', 'create_individual_profiles', 'calculate_percentiles_and_rankings',
    'calculate_adjusted_score', 'create_personalized_description', 'create_user_analysis_data', 'save_all_user_profiles'
]

# 注意：此笔记本已被重写为仅提供数据接口函数，删除所有绘图与交互逻辑。运行者可按需调用这些函数进行处理。


In [67]:
# 绘图相关单元格已移除 — 该单元格保留为空占位，用于兼容性。

In [68]:
# 已移除：交互打印和可视化。相关数据生成功能已迁移到第一个单元格的数据接口函数中。

In [69]:
# 已移除：改进的雷达与绘图函数。请使用第一个单元格中的数据接口函数进行数值计算。

In [70]:
# 已移除：独立对比散点图函数。数据分析请调用 create_user_analysis_data() 等数据接口。

In [71]:
# 已移除：交互性用户选择与主程序。此笔记本现在仅提供函数接口，调用者负责批处理与交互。
